In [ ]:
# Tonight's three, in order of signal per minute.
#
#   1. read the flagged instances       free, 30 min, and the first thing Jake will ask
#   2. divergence on pruned v14         2 min, and the capstone for the talk
#   3. prune-fraction sweep             35 min unattended
#
# No generation, no API calls. Everything runs off verdicts already on disk, so a fresh
# pod only needs the repo and the NFS mount.

import importlib
import json
import random
from collections import Counter
from pathlib import Path

import pandas as pd

import ddi.pruning, ddi.divergence, ddi.verify_binary
for m in (ddi.verify_binary, ddi.pruning, ddi.divergence):
    importlib.reload(m)

from ddi.data import build_human
from ddi.manifest import load_dataset
from ddi.train import train_and_eval
from ddi.experiment import log_run
from ddi.verify_binary import load_verdicts
from ddi.pruning import align, prune, prune_random
from ddi import divergence as dv

V14_ID = "20260807-123340-ff79db"
V15_ID = "20260816-005908-3d6539"

train, dev, val = build_human()
per_sent = Counter(r["sent_id"] for r in train)
train_f = [r for r in train if per_sent[r["sent_id"]] < 190]

v14, _ = load_dataset(V14_ID)
v15, _ = load_dataset(V15_ID)
ver14 = load_verdicts("verify-bin2-v14")
ver15 = load_verdicts("verify-bin2-v15")


# ===========================================================================
# 1. WHAT DID THE VERIFIER REMOVE?
#
# Pruning 1,056 flagged NONEs from v14 gained 0.041 over removing 1,056 at random. The
# gain was recall (0.525 -> 0.623) with precision flat, so these were not mislabelled
# negatives poisoning precision. They were negatives that suppressed firing.
#
# The question is what they have in common. Guess: pairs flanking an asserted pair in
# the same clause. Read fifty and find out, because the answer determines how the whole
# result is framed.
# ===========================================================================
judged14, unjudged14, flagged14 = align(v14, ver14)

flag_ids = {id(r) for r in flagged14}
kept_none = [r for r in judged14 if r["label"] == "NONE" and id(r) not in flag_ids]

import re
MARK = re.compile(r"\[/?E[12]\]")


def describe(rows, name, n=25):
    """Structural profile of a set of NONE instances."""
    per_sent = Counter(r["sent_id"] for r in rows)
    lens = [len(MARK.sub("", r["text"]).split()) for r in rows]
    # does the sentence assert something? (any positive pair from the same sentence)
    print(f"\n{name}: {len(rows)} instances, {len(per_sent)} sentences")
    print(f"  median sentence length {sorted(lens)[len(lens) // 2]}")
    print(f"  mean flagged per sentence {len(rows) / max(len(per_sent), 1):.2f}")


describe(flagged14, "flagged")
describe(kept_none, "kept NONE")

# how concentrated are flags within sentences? if a whole sentence gets flagged, the
# issue is the sentence; if one pair in a sentence does, it is the pair
by_sent_flag = Counter(r["sent_id"] for r in flagged14)
by_sent_all = Counter(r["sent_id"] for r in judged14 if r["label"] == "NONE")
frac = [by_sent_flag[s] / by_sent_all[s] for s in by_sent_flag]
print(f"\nwithin a sentence that has any flag, fraction of its NONEs flagged:")
print(f"  median {sorted(frac)[len(frac) // 2]:.2f}, "
      f"all-flagged {sum(1 for f in frac if f == 1.0)}/{len(frac)}")

# does the flagged sentence also carry a positive pair?
pos_sents = {r["sent_id"] for r in judged14 if r["label"] != "NONE"}
print(f"\nflagged NONEs in a sentence that also asserts: "
      f"{sum(1 for r in flagged14 if r['sent_id'] in pos_sents) / len(flagged14):.3f}")
print(f"kept NONEs in a sentence that also asserts:    "
      f"{sum(1 for r in kept_none if r['sent_id'] in pos_sents) / len(kept_none):.3f}")

print("\n" + "=" * 70)
print("FLAGGED (removed, and removing them helped)")
print("=" * 70)
for r in random.Random(0).sample(flagged14, 25):
    print(f"\n{r['text'][:300]}")

print("\n" + "=" * 70)
print("KEPT NONE (for contrast)")
print("=" * 70)
for r in random.Random(0).sample(kept_none, 15):
    print(f"\n{r['text'][:300]}")


In [ ]:
from collections import defaultdict
by_sent = defaultdict(list)
for r in judged14:
    by_sent[r["sent_id"]].append(r)

for r in random.Random(0).sample(flagged14, 15):
    sib = by_sent[r["sent_id"]]
    print(f"\n{MARK.sub('', r['text'])[:280]}")
    print(f"  FLAGGED: {r['text'][:200]}")
    print(f"  sentence has {len(sib)} pairs: {Counter(x['label'] for x in sib)}")
    for x in sib:
        if x["label"] != "NONE":
            print(f"    {x['label']}: {x['text'][:160]}")

In [ ]:


# ===========================================================================
# 2. DOES THE DIVERGENCE TABLE SEE THE PRUNING?
#
# This is the capstone. The verifier found something worth 0.041 F1. If no divergence
# measurement moves, the framework is blind to the one intervention that worked, which
# is the sharpest statement of the project's central finding. If something moves, it
# partially rescues the tool and names the measurement to trust.
# ===========================================================================
pruned14 = prune(judged14, flagged14)
rand14 = prune_random(judged14, len(flagged14), seed=0)

base = dv.compare(train_f, judged14, match_size=True)
prn = dv.compare(train_f, pruned14, match_size=True)
rnd = dv.compare(train_f, rand14, match_size=True)

tbl = base[["measurement", "tier", "corpus", "synth"]].rename(
    columns={"synth": "judged"})
tbl["pruned"] = prn.synth.values
tbl["random"] = rnd.synth.values
tbl["pruned-judged"] = (tbl.pruned - tbl.judged).round(4)
print("\n" + tbl.to_string(index=False))

print("""
F1: judged 0.387, pruned 0.408, random 0.367.
If pruned-judged is near zero on every row, no measurement in the table predicted the
one intervention that worked, and none of them would have told you to do it.
""")


In [ ]:


# ===========================================================================
# 3. PRUNE-FRACTION SWEEP
#
# One comparison becomes a dose-response curve. Monotone rise is far more convincing
# than a single point, and a peak below 100% is a tuning knob worth reporting.
# ~35 min for both generators.
# ===========================================================================
BASE = {"model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
        "neg_ratio": None, "render_mode": "markers"}

FRACTIONS = [0.0, 0.25, 0.5, 0.75, 1.0]
rows = []

for gen, inst, ver in [("v14", v14, ver14), ("v15", v15, ver15)]:
    judged, _, flagged = align(inst, ver)
    for frac in FRACTIONS:
        n_drop = int(len(flagged) * frac)
        drop = set(id(r) for r in
                   random.Random(0).sample(flagged, n_drop)) if n_drop else set()
        data = [r for r in judged if id(r) not in drop]
        for seed in [0, 1, 2]:
            cfg = {**BASE, "seed": seed, "dataset": f"{gen}-prune{frac:.2f}",
                   "prune_fraction": frac, "n_flagged": len(flagged)}
            m = train_and_eval(cfg, data, dev)
            log_run(cfg, m, notes=f"prune fraction sweep, {gen}, {frac}")
            rows.append({"gen": gen, "frac": frac, "seed": seed, "n": len(data),
                         "f1": m["micro_f1_pos"], "p": m["micro_p_pos"],
                         "r": m["micro_r_pos"]})
            print(f"  {gen} frac={frac:.2f} seed={seed} n={len(data):>6} "
                  f"f1={m['micro_f1_pos']:.3f} r={m['micro_r_pos']:.3f}")

sweep = pd.DataFrame(rows)
Path("runs/prune-sweep.json").write_text(json.dumps(rows, indent=1))
print()
print(sweep.groupby(["gen", "frac"])[["f1", "p", "r"]].agg(["mean", "std"]).to_string())

print("""
Read:
  monotone rise to 1.0   pruning helps and more is better; try ranking the flags and
                         going past 100% by pruning near-misses too
  peak below 1.0         there is an optimum, which is a reportable tuning result
  flat                   the single-point gain was seed noise after all, and the
                         pruned-vs-random contrast needs more seeds before it is claimed
""")